[Reference](https://medium.com/@GaoDalie_AI/forget-loop-engineering-graph-engineering-is-about-this-713a9cf2e985$0)

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# Define the global state structure
class AgentState(TypedDict):
    task: str
    context: list[str]
    draft: str
    quality_score: float
    retry_count: int

# Define each node (each node is a function)
def agent_a_understand(state: AgentState) -> AgentState:
    """Task understanding: convert user input into a structured task"""
    structured_task = llm.invoke(f"Structure the following task: {state['task']}")
    return {"task": structured_task}

def agent_b_retrieve(state: AgentState) -> AgentState:
    """Information retrieval: build a context package"""
    context = rag.search(state["task"], top_k=5)
    return {"context": context, "retry_count": state.get("retry_count", 0)}

def agent_c_generate(state: AgentState) -> AgentState:
    """Core generation: produce a draft based on the context"""
    draft = llm.invoke(build_prompt(state["task"], state["context"]))
    return {"draft": draft}

def agent_d_judge(state: AgentState) -> AgentState:
    """Quality evaluator: score the draft and determine whether it passes"""
    score = evaluator.score(state["draft"], state["task"])
    return {
        "quality_score": score,
        "retry_count": state["retry_count"] + 1
    }

# Conditional edge: D's result determines the next path
def should_retry(state: AgentState) -> str:
    if state["quality_score"] >= 0.85:
        return "end"      # Passed, finish
    if state["retry_count"] >= 3:
        return "end"      # Exceeded maximum retries, force stop
    return "retry"        # Failed, send back for another attempt

# Build the graph
graph = StateGraph(AgentState)
graph.add_node("understand", agent_a_understand)
graph.add_node("retrieve", agent_b_retrieve)
graph.add_node("generate", agent_c_generate)
graph.add_node("judge", agent_d_judge)

# Add edges
graph.set_entry_point("understand")
graph.add_edge("understand", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "judge")

# Conditional edge: judge decides whether to end or retry
# (retry loops back to retrieve)
graph.add_conditional_edges(
    "judge",
    should_retry,
    {
        "end": END,
        "retry": "retrieve"  # Loop back to retrieve on retry
    }
)

app = graph.compile()

result = app.invoke({
    "task": "Analyze the risk clauses in this contract",
    "retry_count": 0
})